# Ablation (Table VII): GAT-only on FER+

Landmark-graph branch only (MediaPipe landmarks -> k-NN graph -> GAT), no image CNN.
Reported in the paper: **34.90% accuracy**.

Note: in the original run this number is the best checkpoint evaluated on the 20% validation split of the FER+ `train` folder (13,276 images).
Its weighted F1 in the original log is 26.9% (the 26.30% in Table VII equals the macro F1 of that log).
Run from the `notebooks/` folder with FER+ placed at `data/FER+`.


In [ ]:
import os
import cv2
import glob
import numpy as np
import pandas as pd
import mediapipe as mp
import torch
import torch.nn as nn
import torch.nn.functional as F
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from torch.utils.data import Dataset, DataLoader, random_split
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
from collections import Counter
warnings.filterwarnings("ignore")

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Configuration
# Base path - update this to your actual path
BASE_DIR = "../data/FER+"
classes = os.listdir(os.path.join(BASE_DIR, "train"))
print(f"Number of classes: {len(classes)}")
print(f"Classes: {classes}")
      
DATA_DIR       = os.path.join(BASE_DIR, "train")
VAL_DIR        = os.path.join(BASE_DIR, "validation")
TEST_DIR       = os.path.join(BASE_DIR, "test")

NUM_CLASSES = 8  # Angry, Happy, Neutral, Sad, Surprise, Ahegao
BATCH_SIZE = 32
EPOCHS = 100
LR = 5e-4
WEIGHT_DECAY = 1e-4
K_NEIGHBORS = 10
EARLY_STOP_PATIENCE = 15

# Create directories for outputs
os.makedirs('../checkpoints', exist_ok=True)
os.makedirs('../outputs/gat_only', exist_ok=True)

# Semantic Subset of 97 Landmarks
LANDMARK_SUBSET_INDICES = [
    # Jawline
    234, 93, 132, 58, 172, 136, 150, 149, 176, 148, 152, 377, 400, 378, 379, 365, 397, 288, 361, 323, 454,
    # Right Eyebrow
    70, 63, 105, 66, 107, 336, 296, 334, 293, 300,
    # Nose Bridge
    168, 6, 197, 195, 5,
    # Nose Bottom
    48, 115, 220, 45, 4, 275, 440, 344, 278,
    # Right Eye
    33, 246, 161, 160, 159, 158, 157, 173, 133, 155, 154, 153, 145, 144, 163, 7,
    # Left Eye
    362, 398, 384, 385, 386, 387, 388, 466, 263, 249, 390, 373, 374, 380, 381, 382,
    # Outer Lip
    0, 267, 269, 270, 409, 291, 375, 321, 405, 314, 17, 84, 181, 91, 146, 61, 185, 40, 39, 37
]
print(f"Selected {len(LANDMARK_SUBSET_INDICES)} landmarks for graph construction.")

# Reproducibility: fix random seeds (the original runs were not seeded, so expect small run-to-run variation)
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
class FaceGraphDataset(Dataset):
    """GAT-only dataset: returns (graph_data, label) — no image tensor."""

    def __init__(self, root_dir, subset_indices=None, k_neighbors=10,
                 subset_fraction=1.0, augment_landmarks=False):
        self.root_dir = root_dir
        self.subset_indices = subset_indices
        self.k_neighbors = k_neighbors
        self.augment_landmarks = augment_landmarks

        # Initialize MediaPipe
        base_options = python.BaseOptions(model_asset_path='../models/face_landmarker.task')
        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            output_face_blendshapes=False,
            output_facial_transformation_matrixes=False,
            num_faces=1
        )
        self.face_mesh = vision.FaceLandmarker.create_from_options(options)

        self.image_paths = []
        self.labels = []
        self.class_map = {}

        self._prepare_dataset(subset_fraction)

    def _prepare_dataset(self, subset_fraction):
        if not os.path.exists(self.root_dir):
            print(f"Path not found: {self.root_dir}")
            return

        classes = sorted([d for d in os.listdir(self.root_dir)
                          if os.path.isdir(os.path.join(self.root_dir, d))])
        print(f"Classes found: {classes}")

        for idx, cls_name in enumerate(classes):
            self.class_map[cls_name] = idx
            cls_path = os.path.join(self.root_dir, cls_name)
            all_imgs = [f for f in os.listdir(cls_path)
                        if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            if subset_fraction < 1.0:
                import random
                random.shuffle(all_imgs)
                all_imgs = all_imgs[:int(len(all_imgs) * subset_fraction)]

            for img_name in all_imgs:
                self.image_paths.append(os.path.join(cls_path, img_name))
                self.labels.append(idx)

        print(f"Total images: {len(self.image_paths)}")
        label_counts = Counter(self.labels)
        print("\nClass distribution:")
        for cls_name, cls_idx in sorted(self.class_map.items(), key=lambda x: x[1]):
            print(f"  {cls_name}: {label_counts[cls_idx]} samples")

    def _procrustes_normalize(self, landmarks):
        """Center and scale landmarks to unit variance."""
        centroid = np.mean(landmarks, axis=0)
        centered = landmarks - centroid
        scale = np.sqrt(np.sum(centered ** 2) / len(centered))
        return (centered / scale if scale > 1e-6 else centered).astype(np.float32)

    def _augment_landmarks(self, landmarks):
        """Add Gaussian noise to landmarks for augmentation."""
        if self.augment_landmarks:
            landmarks = landmarks + np.random.normal(0, 0.01, landmarks.shape)
        return landmarks

    def _get_landmarks(self, image):
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)
        results = self.face_mesh.detect(mp_image)

        if not results.face_landmarks:
            return None

        landmarks = results.face_landmarks[0]
        indices = self.subset_indices if self.subset_indices else range(len(landmarks))

        coords = []
        for idx in indices:
            if idx < len(landmarks):
                lm = landmarks[idx]
                coords.append([lm.x, lm.y, lm.z])

        coords = np.array(coords, dtype=np.float32)
        coords = self._procrustes_normalize(coords)
        coords = self._augment_landmarks(coords)
        return coords

    def _build_graph(self, landmark_coords):
        x = torch.tensor(landmark_coords, dtype=torch.float)

        try:
            nbrs = NearestNeighbors(
                n_neighbors=min(self.k_neighbors + 1, len(landmark_coords)),
                algorithm='ball_tree'
            ).fit(landmark_coords)
            distances, indices = nbrs.kneighbors(landmark_coords)
        except Exception as e:
            print(f"k-NN Error: {e}")
            return None

        edge_index = []
        edge_attr = []
        for i in range(len(landmark_coords)):
            for j_idx in range(1, len(indices[i])):
                j = indices[i, j_idx]
                dist = distances[i, j_idx]
                edge_index.append([i, j])
                edge_attr.append([dist])
                edge_index.append([j, i])
                edge_attr.append([dist])

        if not edge_index:
            edge_index = [[i, i] for i in range(len(landmark_coords))]
            edge_attr = [[0.0] for _ in range(len(landmark_coords))]

        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = cv2.imread(img_path)
        if image is None:
            return self.__getitem__((idx + 1) % len(self))

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        landmarks = self._get_landmarks(image_rgb)
        if landmarks is None:
            return self.__getitem__((idx + 1) % len(self))

        graph_data = self._build_graph(landmarks)
        if graph_data is None:
            return self.__getitem__((idx + 1) % len(self))

        return graph_data, torch.tensor(label, dtype=torch.long)

In [ ]:
def collate_fn(batch):
    graphs, labels = zip(*batch)
    labels = torch.stack(labels)
    graphs = Batch.from_data_list(graphs)
    return graphs, labels

In [ ]:
class GATOnlyFaceGNN(nn.Module):
    """Pure GAT model for face emotion recognition.
    Input: landmark graph  →  4 GAT layers  →  dual pooling  →  classifier.
    """

    def __init__(self, num_node_features=3, num_classes=6):
        super(GATOnlyFaceGNN, self).__init__()

        # --- GAT layers ---
        self.gat1 = GATConv(num_node_features, 128, heads=4, concat=True, dropout=0.3)
        self.bn1  = nn.BatchNorm1d(128 * 4)

        self.gat2 = GATConv(128 * 4, 256, heads=4, concat=True, dropout=0.3)
        self.bn2  = nn.BatchNorm1d(256 * 4)

        self.gat3 = GATConv(256 * 4, 512, heads=2, concat=True, dropout=0.3)
        self.bn3  = nn.BatchNorm1d(512 * 2)

        self.gat4 = GATConv(512 * 2, 512, heads=1, concat=False, dropout=0.3)
        self.bn4  = nn.BatchNorm1d(512)

        # Dual pooling: mean + max  →  1024-dim graph embedding
        graph_dim = 512 * 2

        # --- Classifier ---
        self.classifier = nn.Sequential(
            nn.Linear(graph_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, graph_data):
        x, edge_index, batch = graph_data.x, graph_data.edge_index, graph_data.batch

        # GAT layers
        x = F.elu(self.bn1(self.gat1(x, edge_index)))
        x = F.elu(self.bn2(self.gat2(x, edge_index)))
        x = F.elu(self.bn3(self.gat3(x, edge_index)))
        x = F.elu(self.bn4(self.gat4(x, edge_index)))

        # Dual pooling
        graph_emb = torch.cat([global_mean_pool(x, batch),
                                global_max_pool(x, batch)], dim=1)

        return self.classifier(graph_emb)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel will have approximately {count_parameters(GATOnlyFaceGNN()):,} trainable parameters")

In [ ]:
class LabelSmoothingCrossEntropy(nn.Module):
    """Cross entropy with label smoothing."""

    def __init__(self, epsilon=0.1, weight=None):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.epsilon = epsilon
        self.weight = weight

    def forward(self, outputs, targets):
        n_classes = outputs.size(-1)
        log_probs = F.log_softmax(outputs, dim=-1)

        targets_one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_smooth = (1 - self.epsilon) * targets_one_hot + self.epsilon / n_classes

        loss = -(targets_smooth * log_probs).sum(dim=-1)

        if self.weight is not None:
            loss = loss * self.weight[targets]

        return loss.mean()

In [ ]:
def train(model, loader, criterion, optimizer, scheduler, device, epoch):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for graphs, labels in loader:
        graphs = graphs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(graphs)
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    scheduler.step()
    return total_loss / len(loader), 100. * correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for graphs, labels in loader:
            graphs = graphs.to(device)
            labels = labels.to(device)

            outputs = model(graphs)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), 100. * correct / total, all_preds, all_labels

In [ ]:
print("Initializing dataset with landmark augmentation...")
full_dataset = FaceGraphDataset(
    root_dir=DATA_DIR,
    subset_indices=LANDMARK_SUBSET_INDICES,
    k_neighbors=K_NEIGHBORS,
    augment_landmarks=True
)

# Split dataset
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Data loaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=0
)

print(f"\nTrain samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

# Compute class weights for handling imbalance
train_labels = [full_dataset.labels[i] for i in train_dataset.indices]
class_counts = Counter(train_labels)
total_samples = len(train_labels)
class_weights = torch.tensor(
    [total_samples / (NUM_CLASSES * class_counts[i]) for i in range(NUM_CLASSES)],
    dtype=torch.float
).to(device)

print(f"\nClass weights for balanced training: {class_weights.cpu().numpy()}")

In [ ]:
# Model initialization
print("\nInitializing GAT-only model...")
model = GATOnlyFaceGNN(num_node_features=3, num_classes=NUM_CLASSES).to(device)
print(f"Total parameters: {count_parameters(model):,}")

# Loss function with label smoothing and class weights
criterion = LabelSmoothingCrossEntropy(epsilon=0.1, weight=class_weights)

# Optimizer with weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Cosine annealing scheduler with warm restarts
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# Training history
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [],  'val_acc': [],
    'learning_rates': []
}

best_val_acc = 0.0
patience_counter = 0

print("\n" + "="*80)
print("Starting training (GAT-only architecture)...")
print("="*80)

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, scheduler, device, epoch)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['learning_rates'].append(optimizer.param_groups[0]['lr'])

    print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:6.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:6.2f}% | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'history': history
        }, '../checkpoints/gat_only_ferplus.pth')
        print(f"  → New best model saved! Val Acc: {val_acc:.2f}%")
    else:
        patience_counter += 1

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered after {epoch+1} epochs.")
        print(f"Best validation accuracy: {best_val_acc:.2f}%")
        break

print("\n" + "="*80)
print(f"Training completed! Best validation accuracy: {best_val_acc:.2f}%")
print("="*80)

with open('../outputs/gat_only/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

In [ ]:
# Load best model for final evaluation
print("Loading best model for final evaluation...")
checkpoint = torch.load('../checkpoints/gat_only_ferplus.pth')
model.load_state_dict(checkpoint['model_state_dict'])

# Final evaluation
val_loss, val_acc, all_preds, all_labels = evaluate(model, val_loader, criterion, device)

print(f"\nFinal Validation Results:")
print(f"  Loss: {val_loss:.4f}")
print(f"  Accuracy: {val_acc:.2f}%")

# Classification report
class_names = sorted(full_dataset.class_map.keys(), key=lambda x: full_dataset.class_map[x])
print("\nPer-Class Performance:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss curves
axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history['val_loss'],   label='Val Loss',   linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# Accuracy curves
axes[0, 1].plot(history['train_acc'], label='Train Acc', linewidth=2)
axes[0, 1].plot(history['val_acc'],   label='Val Acc',   linewidth=2)
axes[0, 1].axhline(y=90, color='r', linestyle='--', label='Target (90%)', linewidth=1.5)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
axes[0, 1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# Learning rate schedule
axes[1, 0].plot(history['learning_rates'], linewidth=2, color='green')
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Learning Rate', fontsize=12)
axes[1, 0].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True, alpha=0.3)

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1, 1], cbar_kws={'label': 'Count'})
axes[1, 1].set_xlabel('Predicted', fontsize=12)
axes[1, 1].set_ylabel('True', fontsize=12)
axes[1, 1].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/gat_only/training_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTraining visualizations saved to '../outputs/gat_only/training_results.png'")

In [ ]:
# Final Summary
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)
print(f"\nModel Architecture: GAT-Only (4-layer GAT + dual pooling + classifier)")
print(f"Total Parameters: {count_parameters(model):,}")
print(f"\nTraining Configuration:")
print(f"  - Epochs: {len(history['train_acc'])}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Initial Learning Rate: {LR}")
print(f"  - Weight Decay: {WEIGHT_DECAY}")
print(f"  - k-Neighbors: {K_NEIGHBORS}")
print(f"\nFinal Results:")
print(f"  - Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"  - Final Training Accuracy: {history['train_acc'][-1]:.2f}%")
print(f"  - Train-Val Gap: {abs(history['train_acc'][-1] - best_val_acc):.2f}%")

if best_val_acc >= 90:
    print(f"\n✅ SUCCESS: Target accuracy of 90% achieved!")
elif best_val_acc >= 85:
    print(f"\n⚠️  CLOSE: Achieved {best_val_acc:.2f}%, just below 90% target.")
    print(f"   Consider: longer training, larger batch size, or more data augmentation.")
else:
    print(f"\n❌ BELOW TARGET: Achieved {best_val_acc:.2f}%, target is 90%.")
    print(f"   Recommendations:")
    print(f"   - Increase model capacity (more layers/channels)")
    print(f"   - Add more aggressive landmark augmentation")
    print(f"   - Check for data quality issues")

print("\n" + "="*80)